# 🚁 RT-DETR Drone Detection (Full Pipeline)

**Model:** RT-DETR-L (Transformer)
**Hardware:** Tesla T4 (Google Colab)
**Dataset:** cybersimar08/drone-detection

---

### 1. Setup Environment & Kaggle API

In [ ]:
# Install Dependencies
!pip install -q ultralytics kaggle

import os
from google.colab import files

# Upload kaggle.json
if not os.path.exists('/content/kaggle.json'):
    print("📂 Please upload your 'kaggle.json' file now:")
    uploaded = files.upload()
    
    # Move to correct location
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("✅ Kaggle API configured.")
else:
    print("✅ Kaggle API already configured.")

### 2. Download Dataset

In [ ]:
# Download the specific dataset
if not os.path.exists('/content/dataset'):
    print("⬇️ Downloading dataset (cybersimar08/drone-detection)...")
    !kaggle datasets download -d cybersimar08/drone-detection
    print("📦 Unzipping...")
    !unzip -q drone-detection.zip -d /content/dataset
    print("✅ Dataset Ready.")
else:
    print("✅ Dataset already exists.")

### 3. Configure Paths (Fix YAML)

In [ ]:
import yaml
import glob

# Find the data.yaml file automatically
yaml_files = glob.glob('/content/dataset/**/data.yaml', recursive=True)
if not yaml_files:
    raise FileNotFoundError("Could not find data.yaml in the dataset!")

yaml_path = yaml_files[0]
dataset_root = os.path.dirname(yaml_path)
print(f"📍 Found config at: {yaml_path}")

# Update paths to be absolute for Colab
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

data_config['path'] = dataset_root
data_config['train'] = 'train/images'
data_config['val'] = 'valid/images'
data_config['test'] = 'test/images'

# Write back the fixed yaml
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)
print("✅ data.yaml paths fixed for Colab.")

### 4. Train RT-DETR Model
*Note: Using Batch Size 8 for T4 GPU stability.*

In [ ]:
from ultralytics import RTDETR

# Load RT-DETR-L (Large) Model
model = RTDETR('rtdetr-l.pt') 

print("🔥 STARTING TRAINING (30 Epochs)...")

# Train
results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=640,
    batch=8,             # Optimized for T4 GPU (16GB VRAM)
    device=0,
    workers=2,
    project='/content/runs/train',
    name='rtdetr_drone',
    exist_ok=True,
    amp=True             # Mixed Precision enabled
)

### 5. Validate & Display Predictions

In [ ]:
from IPython.display import Image, display
import glob

print("📊 Generating Predictions on Test Images...")

# Pick a random image from the test set to display
test_images = glob.glob(f"{dataset_root}/test/images/*.jpg")

if test_images:
    # Run prediction on one image
    results_pred = model.predict(test_images[0], save=True, conf=0.5)
    
    # Show the result
    print(f"\n👁️ Prediction Result: {test_images[0]}")
    pred_path = results_pred[0].save_dir + '/' + os.path.basename(test_images[0])
    display(Image(filename=pred_path, width=600))
    
    # Also show the training results graph
    results_png = '/content/runs/train/rtdetr_drone/results.png'
    if os.path.exists(results_png):
        print("\n📈 Training Metrics:")
        display(Image(filename=results_png, width=800))
else:
    print("⚠️ No test images found to display.")

### 6. Export & Download Model

In [ ]:
import shutil
from google.colab import files

print("📦 Zipping model and logs for download...")
shutil.make_archive('/content/rtdetr_drone_model', 'zip', '/content/runs/train/rtdetr_drone')

print("⬇️ Downloading file...")
files.download('/content/rtdetr_drone_model.zip')